Auf X wurde von Chris Luening ein mögliches Zielszenario für den EE Ausbau vorgestellt. 
https://x.com/DerClue/status/2034684857981235478

Man baue für 10.000 Euro/kW
(Akw im Westen 7000-12000 USD/kW)

1 Gaskraftwerk 500-700 Euro/kW
3 PV Parks je 700 Euro / kW
2 Windparks je 1700 Euro/kW
Speicher für 12h = 2400 Euro
Netzausbaukosten 400 Euro/kW
Lyseur 1000 Euro/kW
Das kW bezieht sich hier auf die gesicherte Leistung. 

CF für neue Windparks und Modelle wie E175, V172, N175 in D 20-40%
2100-3500h (belegbar mit Referenzerträgen, die für diese Anlagen eintrudeln im Rahmen EEG-Standortgütefaktor). 

Dieses Szenario lasse ich gegen 2024+2025 laufen.
1 kW Gaskraftwerk (durch Wasserstoff betrieben)
3 kWp PV (1000 kWh/kWp)
2 kW Wind (onshore 3 Szenarien 2100 kWh/kW - 2800 kWh/kW und 3500 kWh/kW)
1 kW Anschlussleistung Batterie (12 kWh Speicher)
1 kWel Elektrolyseur
Gasspeicher ist unendlich im Szenario. 

Die elektrische Leistung wird auf den höchsten Tagesdurchschnitt vom Stromverbrauch in Deutschland normiert. 
Pumpspeicherkraftwerke, Ausland, Biomasse und andere Faktoren werden nicht betrachtet. 

Folgende Variablen sind somit über das Jahr vorhanden: 
Produktion PV/Wind
Einspeicherung Akku, Elektrolyseur
Nutzung Wasserstoff/Akku. 

Die Daten kommen von der Bundesnetzagentur (smard.de)

In [ ]:
In der ersten Iteration wird ohne Wetterprognose und einem einfachen Batterieeinsatz gerechnet. Die Batterie hat keine Ladebeschränkung.


In [100]:
import pandas as pd
import numpy as np

jahr = '2025'

#if jahr == '2025':
ghp="Daten entsoe/Gro_handelspreise_202311010000_202601010000_Viertelstunde.csv"
psf="Daten entsoe/Physikalischer_Stromfluss_202311010000_202601010000_Viertelstunde.csv"
kma="Daten entsoe/Kommerzieller_Au_enhandel_202311010000_202601010000_Viertelstunde.csv"
rez='Daten entsoe/Realisierte_Erzeugung_202311010000_202601010000_Viertelstunde.csv'
rsv='Daten entsoe/Realisierter_Stromverbrauch_202311010000_202502010000_Viertelstunde.csv'
#else:
#    ghp="Daten entsoe/Gro_handelspreise_202311010000_202601020000_Stunde_Viertelstunde.csv"
#    psf="Daten entsoe/Physikalischer_Stromfluss_202311010000_202601020000_Viertelstunde.csv"
#    rez='Daten entsoe/Realisierte_Erzeugung_202311010000_202601020000_Viertelstunde.csv'

Strompreis=pd.read_csv(ghp,na_values='-',header=0 ,sep=';', decimal=",", thousands=".")

In [101]:
Strompreis['Datum']=pd.to_datetime(Strompreis['Datum von'],format='%d.%m.%Y %H:%M')
Strompreis=Strompreis.sort_values('Datum',kind='stable')
Strompreis=Strompreis[Strompreis['Datum von'].str[14:16]=='00']
Strompreis['Datum']=pd.to_datetime(Strompreis['Datum von'],format='%d.%m.%Y %H:%M')
Strompreis['Datum']=Strompreis['Datum'].dt.tz_localize('Europe/Berlin',ambiguous='infer')
Strompreis['Date']=Strompreis['Datum'].dt.tz_convert('UTC')
Strompreis=Strompreis.set_index('Date')

In [102]:
datei=kma
dtype={'Datum von':"string",'Datum bis':"string"
       ,'Nettoexport [MWh] Originalauflösungen':np.float64
       ,'Niederlande (Export) [MWh] Originalauflösungen':np.float64
       ,'Niederlande (Import) [MWh] Originalauflösungen':np.float64
       ,'Schweiz (Export) [MWh] Originalauflösungen':np.float64
       ,'Schweiz (Import) [MWh] Originalauflösungen':np.float64
       ,'Dänemark (Export) [MWh] Originalauflösungen':np.float64
       ,'Dänemark (Import) [MWh] Originalauflösungen':np.float64
       ,'Tschechien (Export) [MWh] Originalauflösungen':np.float64
       ,'Tschechien (Import) [MWh] Originalauflösungen':np.float64
       ,'Luxemburg (Export) [MWh] Originalauflösungen':np.float64
       ,'Luxemburg (Import) [MWh] Originalauflösungen':np.float64
       ,'Schweden (Export) [MWh] Originalauflösungen':np.float64
       ,'Schweden (Import) [MWh] Originalauflösungen':np.float64
       ,'Österreich (Export) [MWh] Originalauflösungen':np.float64
       ,'Österreich (Import) [MWh] Originalauflösungen':np.float64
       ,'Frankreich (Export) [MWh] Originalauflösungen':np.float64
       ,'Frankreich (Import) [MWh] Originalauflösungen':np.float64
       ,'Polen (Export) [MWh] Originalauflösungen':np.float64
       ,'Polen (Import) [MWh] Originalauflösungen':np.float64
       ,'Norwegen (Export) [MWh] Originalauflösungen':np.float64
       ,'Norwegen (Import) [MWh] Originalauflösungen':np.float64
       ,'Belgien (Export) [MWh] Originalauflösungen':np.float64
       ,'Belgien (Import) [MWh] Originalauflösungen':np.float64}


#Stromfluss=pd.read_csv("Daten entsoe/Physikalischer_Stromfluss_202311010000_202502010000_Viertelstunde.csv",sep=';', decimal=",", thousands=".",dtype=dtype,on_bad_lines='skip')
Stromfluss=pd.read_csv(datei,sep=';', decimal=",", thousands=".",header=0,na_values='-',on_bad_lines='skip')

In [107]:
datei=rez
Stromerzeugung=pd.read_csv(datei,sep=';', decimal=",", thousands=".",header=0,na_values='-',on_bad_lines='skip')

In [108]:
datei=rsv
Stromverbrauch=pd.read_csv(datei,sep=';', decimal=",", thousands=".",header=0,na_values='-',on_bad_lines='skip')

In [137]:
Strom= Stromverbrauch.copy()
Strom['Wind']=Stromerzeugung['Wind Onshore [MWh] Originalauflösungen']
Strom['PV']=Stromerzeugung['Photovoltaik [MWh] Originalauflösungen']
Strom.drop('Datum bis', axis=1, inplace=True)
Strom.drop('Residuallast [MWh] Originalauflösungen', axis=1, inplace=True)
Strom.drop('Pumpspeicher [MWh] Originalauflösungen', axis=1, inplace=True)


In [138]:
Strom['Datum']=pd.to_datetime(Strom['Datum von'],format='%d.%m.%Y %H:%M')
#Strom=Strom.sort_values('Datum',kind='stable')
Strom['Datum']=Strom['Datum'].dt.tz_localize('Europe/Berlin',ambiguous='infer')
Strom['Date']=Strom['Datum'].dt.tz_convert('UTC')
Strom=Strom.set_index('Date')
Strom.drop('Datum von', axis=1, inplace=True)
Strom.drop('Datum', axis=1, inplace=True)
Strom.rename(columns={'Gesamt (Netzlast) [MWh] Originalauflösungen':'Netzlast'}, inplace=True)

In [139]:
Strom=Strom[Strom.index.year==2024]

In [140]:
pv_kwh_je_kwp=1000
wind_kwh_je_kw=2800

pv_kWp_je_verbrauch=3
wind_kW_je_verbrauch=2


max_verbrauch_tag=Strom.groupby(Strom.index.date).agg(['sum'])['Netzlast'].max().min()/24
sum_solar_power=Strom['PV'].sum().min()
sum_wind_power=Strom['Wind'].sum().min()

pv_kWp_in_de=sum_solar_power/pv_kwh_je_kwp
win_kW_in_de=sum_wind_power/wind_kwh_je_kw


faktor_pv=max_verbrauch_tag/pv_kWp_in_de*pv_kWp_je_verbrauch
faktor_wind=max_verbrauch_tag/win_kW_in_de*wind_kW_je_verbrauch


Der höchste Tagesdurchschnitt von 2024 war 66 MW. 
PV =  66289.260417 * 3 = ~ 200 GWp PV sein. 
Wind = 66289.260417 * 2 = ~ 

In [141]:
sum_solar_power,sum_wind_power,max_verbrauch_tag,faktor_pv,faktor_wind,pv_kWp_in_de,win_kW_in_de

(np.float64(63444109.0),
 np.float64(113029839.0),
 np.float64(66289.26041666667),
 np.float64(3.134535016481988),
 np.float64(3.2842642404660363),
 np.float64(63444.109),
 np.float64(40367.79964285714))

In [165]:
Strom_normiert=Strom.copy()
Strom_normiert['PV']=faktor_pv*Strom['PV']/max_verbrauch_tag
Strom_normiert['Wind']=faktor_wind*Strom['Wind']/max_verbrauch_tag
Strom_normiert['Netzlast']=Strom['Netzlast']/max_verbrauch_tag

Strom_normiert['delta']=Strom_normiert['PV']+Strom_normiert['Wind']-Strom_normiert['Netzlast']
batt_speicherstand=0
h2_speicherstand=10000
batt_max=12
h2_max=999999999

a_b_speicherstand=[]
a_h2_speicherstand=[]
a_b_speicherverbrauch=[]
a_h2_speicherverbrauch=[]
a_fehlt_speicher=[]

speicherwk_bat=0.9
speicherwk_h2=0.5
max_batt_speicher_rel=0.25
max_h2_speicher_rel=0.25

for change_speicher in Strom_normiert['delta']:
    speicher_b_alt=batt_speicherstand
    speicher_h2_alt=h2_speicherstand

# batterie
    if change_speicher > 0:
        noch_speicher=batt_max-speicher_b_alt
        b_change_sp=change_speicher*speicherwk_bat
        n_speicher=max(min(b_change_sp,max_batt_speicher_rel),noch_speicher)
        batt_speicherstand=speicher_b_alt+n_speicher
        change_speicher=change_speicher-n_speicher/speicherwk_bat

# h2

    if change_speicher > 0:
        noch_speicher=h2_max-speicher_h2_alt
        h2_change_sp=change_speicher*speicherwk_h2
        n_speicher=max(min(h2_change_sp,max_h2_speicher_rel),noch_speicher)
        h2_speicherstand=speicher_h2_alt+n_speicher
        change_speicher=change_speicher-n_speicher/speicherwk_bat

    nutz_b=0
    if change_speicher <= 0:
        nutz_b=max(max_batt_speicher_rel,speicher_b_alt)
        batt_speicherstand=speicher_b_alt-nutz_b
        b_change_sp=-nutz_b
        change_speicher=change_speicher-nutz_b

    nutz_h2=0
    if change_speicher <= 0:
        nutz_h2=max(max_batt_speicher_rel,speicher_h2_alt)
        h2_speicherstand=speicher_h2_alt-nutz_h2
        change_speicher=change_speicher-nutz_h2
        h2_change_sp=-nutz_h2

    fehlt=0
    if change_speicher <= 0:
        fehlt=change_speicher
    
    
    a_b_speicherstand.append(batt_speicherstand)
    a_b_speicherverbrauch.append(nutz_b)

    a_h2_speicherstand.append(h2_speicherstand)
    a_h2_speicherverbrauch.append(nutz_h2)
    a_fehlt_speicher.append(fehlt)

Strom_normiert['Batterie']=a_b_speicherstand
Strom_normiert['Batterie verbrauch']=a_b_speicherverbrauch
Strom_normiert['H2-Speicher']=a_h2_speicherstand
Strom_normiert['H2 verbrauch']=a_h2_speicherverbrauch
Strom_normiert['Ohne Speicher']=a_fehlt_speicher



In [166]:
Strom_normiert

,Netzlast,Wind,PV,delta,Batterie,Batterie verbrauch,H2-Speicher,H2 verbrauch,Ohne Speicher
Date,,,,,,,,,
2024-01-01 00:00:00+00:00,0.148354,0.370035,0.000035,0.221717,-0.25,0.25,0.00,10000.00,-10013.361616
2024-01-01 00:15:00+00:00,0.147049,0.367954,0.000035,0.220941,-0.50,0.25,-0.25,0.25,-13.890170
2024-01-01 00:30:00+00:00,0.145634,0.364932,0.000035,0.219333,-0.75,0.25,-0.50,0.25,-14.169556
2024-01-01 00:45:00+00:00,0.144552,0.366592,0.000035,0.222075,-1.00,0.25,-0.75,0.25,-14.444592
2024-01-01 01:00:00+00:00,0.144850,0.367471,0.000035,0.222657,-1.25,0.25,-1.00,0.25,-14.721788
...,...,...,...,...,...,...,...,...,...
2024-12-31 22:45:00+00:00,0.183838,0.399217,0.000071,0.215449,-8783.00,0.25,-8782.75,0.25,-9772.228995
2024-12-31 23:00:00+00:00,0.179573,0.408420,0.000118,0.228965,-8783.25,0.25,-8783.00,0.25,-9772.493257
2024-12-31 23:15:00+00:00,0.179592,0.406673,0.000106,0.227188,-8783.50,0.25,-8783.25,0.25,-9772.772812


from IPython.display import Markdown as md
pv_kWp_je_verbrauch is {{pv_kWp_je_verbrauch}}

In [160]:
Strom_normiert.to_csv('test.csv')

In [155]:
Strom_normiert['bat']=a_b_speicherstand

In [156]:
Strom_normiert

,Netzlast,Wind,PV,delta,Batterie,delta Batterie,H2-Speicher,delta H2,bat
Date,,,,,,,,,
2024-01-01 00:00:00+00:00,0.148354,0.370035,0.000035,0.221717,0,0,0,0,0.221717
2024-01-01 00:15:00+00:00,0.147049,0.367954,0.000035,0.220941,0,0,0,0,0.442658
2024-01-01 00:30:00+00:00,0.145634,0.364932,0.000035,0.219333,0,0,0,0,0.661991
2024-01-01 00:45:00+00:00,0.144552,0.366592,0.000035,0.222075,0,0,0,0,0.884066
2024-01-01 01:00:00+00:00,0.144850,0.367471,0.000035,0.222657,0,0,0,0,1.106722
...,...,...,...,...,...,...,...,...,...
2024-12-31 22:45:00+00:00,0.183838,0.399217,0.000071,0.215449,0,0,0,0,12.000000
2024-12-31 23:00:00+00:00,0.179573,0.408420,0.000118,0.228965,0,0,0,0,12.000000
2024-12-31 23:15:00+00:00,0.179592,0.406673,0.000106,0.227188,0,0,0,0,12.000000
